In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [3]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,15),
#                     dates < datetime(2026,9,6)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,20),
                     dates < datetime(2026,10,6)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,15),
#                     dates < datetime(2026,10,6)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260820T002003_V202610201927C_0648200021.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20261005T093003_V202610211201C_0650050504.fits'),
 134)

In [4]:
Q = []
dates = []

for file in files[80:]:
    with fits.open(file) as hdul:
        header = hdul[0].header
        data = hdul[0].data
    header['CROTA'] += 0.25

    data_, view = reproject(data, header, kind='synoptic', correct_dr=True, car_rot=2)
    #show_data(data_, view, cmap='seismic', vmin=-50, vmax=50, to_folder='png')

    Q += [data_[600:,:150]]
    dates += [datetime.fromisoformat(header['DATE-OBS'])]

Q = np.array(Q)
dates = np.array(dates)

In [45]:
import matplotlib.dates as mdates

dt = np.array([(date - dates[0]).total_seconds() / 3600 for date in dates])

q = np.nanmean(Q[:,:,20:140], axis=-1).T
nx, ny = q.shape

yi = np.arange(0,ny) / ny * np.max(dt)
p = np.searchsorted(dt, yi, side='right')
a = yi - dt[(p - 1).clip(0)]
b = dt[p.clip(0,ny - 1)] - yi
q_ = q[:,(p - 1).clip(0)] * b / (a + b + 1e-16) + a / (a + b + 1e-16) * q[:, p.clip(0,ny - 1)]

fig, ax = plt.subplots(figsize=(10,8))
ax.imshow(q_, cmap='seismic', vmin=-25, vmax=25, origin='lower', aspect='auto', extent=[dates[0], dates[-1], 60, 90])

ax.set_ylabel('Latitude, degrees')

ax.set_xticklabels(dates)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
fig.autofmt_xdate()
plt.tight_layout()

/tmp/ipykernel_284601/3404572818.py:19: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(dates)


In [40]:
700000 * 0.4 * np.pi / 180 / 24 / 3600

0.05656159612944586